In [2]:
import pandas as pd
import numpy as np
import os
import warnings

warnings.filterwarnings('ignore')

def gerar_metricas_markowitz_capm_v2(diretorio_dados):
    print("=== INÍCIO DO CÁLCULO DE PARÂMETROS GLOBAIS (MARKOWITZ & CAPM) ===")
    
    # 1. Carregamento da Matriz de Prémio de Risco
    caminho_entrada = os.path.join(diretorio_dados, "matriz_premio_risco.csv")
    print("1. A carregar vetores de retornos em excesso (Prémio de Risco)...")
    df_er = pd.read_csv(caminho_entrada, index_col='Data', parse_dates=True)
    
    # Separar os ativos do mercado (IBOV)
    colunas_ativos = [col for col in df_er.columns if col != 'Premio_Mercado_IBOV']
    df_ativos_er = df_er[colunas_ativos]
    er_mercado = df_er['Premio_Mercado_IBOV']
    
    # ---------------------------------------------------------
    # FASE 1: MATRIZ DE COVARIÂNCIA ANUALIZADA
    # ---------------------------------------------------------
    print("2. A processar a Matriz de Covariância Anualizada...")
    # Multiplicamos por 252 (dias úteis) para anualizar a matriz
    matriz_covariancia = df_ativos_er.cov() * 252
    
    # ---------------------------------------------------------
    # FASE 2: CÁLCULO DO BETA E MÉTRICAS DE RISCO
    # ---------------------------------------------------------
    print("3. A derivar os Coeficientes Beta e Índices de Sharpe de forma vetorial...")
    
    # Beta = Covariância(Ativo, Mercado) / Variância(Mercado)
    variancia_mercado_diaria = er_mercado.var()
    covariancia_com_mercado_diaria = df_ativos_er.apply(lambda x: x.cov(er_mercado))
    betas = covariancia_com_mercado_diaria / variancia_mercado_diaria
    
    # Retornos e Volatilidades Anualizadas
    media_er_anualizada = df_ativos_er.mean() * 252
    volatilidade_anualizada = df_ativos_er.std() * np.sqrt(252)
    
    # Índice de Sharpe (Retorno em Excesso / Volatilidade)
    # Como já é o retorno em excesso, não precisamos de subtrair o CDI de novo!
    sharpe_ratios = media_er_anualizada / volatilidade_anualizada
    
    # ---------------------------------------------------------
    # FASE 3: CONSOLIDAÇÃO DOS KPIs
    # ---------------------------------------------------------
    df_kpis = pd.DataFrame({
        'Retorno_Excesso_Anualizado': media_er_anualizada,
        'Volatilidade_Anualizada': volatilidade_anualizada,
        'Beta_CAPM': betas,
        'Indice_Sharpe': sharpe_ratios
    })
    
    # ---------------------------------------------------------
    # FASE 4: EXPORTAÇÃO
    # ---------------------------------------------------------
    print("4. A exportar arquitetura relacional para o diretório...")
    caminho_cov = os.path.join(diretorio_dados, "matriz_covariancia_anualizada.csv")
    caminho_kpis = os.path.join(diretorio_dados, "kpis_ativos_markowitz.csv")
    
    matriz_covariancia.to_csv(caminho_cov)
    df_kpis.to_csv(caminho_kpis)
    
    print("\n=== RESUMO DOS PARÂMETROS INFERENCIAIS ===")
    print(f"Dimensão da Matriz de Covariância: {matriz_covariancia.shape}")
    print(f"Beta Médio da Amostra: {betas.mean():.4f}")
    print(f"Média do Retorno em Excesso (Anualizado): {media_er_anualizada.mean():.4f}")
    print(f"Índice de Sharpe Médio: {sharpe_ratios.mean():.4f}")
    print("==================================================================")
    
    return matriz_covariancia, df_kpis

# ==========================================
# ÁREA DE EXECUÇÃO
# ==========================================
pasta_dados = r"C:\VSCodeWorkspace\TCC_Escrito\data"

# Executar a função
cov_matrix, kpis = gerar_metricas_markowitz_capm_v2(pasta_dados)

=== INÍCIO DO CÁLCULO DE PARÂMETROS GLOBAIS (MARKOWITZ & CAPM) ===
1. A carregar vetores de retornos em excesso (Prémio de Risco)...
2. A processar a Matriz de Covariância Anualizada...
3. A derivar os Coeficientes Beta e Índices de Sharpe de forma vetorial...
4. A exportar arquitetura relacional para o diretório...

=== RESUMO DOS PARÂMETROS INFERENCIAIS ===
Dimensão da Matriz de Covariância: (136, 136)
Beta Médio da Amostra: 0.8057
Média do Retorno em Excesso (Anualizado): 0.0407
Índice de Sharpe Médio: 0.1114
